# Stage 3 DINOv2 Train-CV Policy Clean

Train-CV-selected DINOv2 coarse-only policy on clean GT crops.


In [ ]:

from pathlib import Path
import json, shutil, subprocess
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")
RUN_NAME = "stage3_dinov2_traincv_policy_clean"
ARCHIVE_PATH = Path("/kaggle/working/stage3_deliverables_dinov2_traincv_policy_clean.tar.gz")
DATASET_ROOT_CANDIDATES = [Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"), Path("/kaggle/input/idid-coco-v3")]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
FEATURE_MODEL_ID = "facebook/dinov2-base"
FEATURE_LOGREG_C = 0.03
print("RUN_NAME:", RUN_NAME)


In [ ]:

def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout: print(p.stdout)
    if p.stderr: print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path):
    rows=[]
    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

def pick_data_root():
    for root in DATASET_ROOT_CANDIDATES:
        if (root/TRAIN_JSONL_REL).exists() and (root/VAL_JSONL_REL).exists():
            return root
    raise FileNotFoundError("stage3_regrouped_v2 train/val jsonl not found")


In [ ]:

gpu=sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required")
DATA_ROOT=pick_data_root()
train_jsonl=DATA_ROOT/TRAIN_JSONL_REL
val_jsonl=DATA_ROOT/VAL_JSONL_REL
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])
sh(["python", "-m", "pip", "install", "-q", "transformers==4.57.1", "accelerate", "scikit-learn", "pyyaml", "tabulate"], cwd=REPO_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("train_jsonl:", train_jsonl)
print("val_jsonl:", val_jsonl)


In [ ]:

import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoImageProcessor

train_rows=[r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
val_rows=[r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in LABELS]
print("train rows:", len(train_rows), Counter(r["coarse_class"] for r in train_rows))
print("val rows:", len(val_rows), Counter(r["coarse_class"] for r in val_rows))

def resolve_crop(row, jsonl_path):
    p=Path(row["crop_path"])
    if p.is_absolute() and p.exists(): return p
    for c in [jsonl_path.parent/p, jsonl_path.parent.parent/p, DATA_ROOT/p]:
        if c.exists(): return c
    raise FileNotFoundError(row["crop_path"])

device="cuda" if torch.cuda.is_available() else "cpu"
processor=AutoImageProcessor.from_pretrained(FEATURE_MODEL_ID)
model=AutoModel.from_pretrained(FEATURE_MODEL_ID).to(device)
model.eval()

def embed(paths, batch_size=24):
    feats=[]
    with torch.no_grad():
        for start in range(0, len(paths), batch_size):
            batch_paths=paths[start:start+batch_size]
            images=[Image.open(p).convert("RGB") for p in batch_paths]
            inputs=processor(images=images, return_tensors="pt").to(device)
            out=model(**inputs)
            feat=out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state.mean(dim=1)
            feat=feat.float(); feat=feat/feat.norm(dim=-1, keepdim=True)
            feats.append(feat.cpu().numpy())
            print("embedded", min(start+batch_size, len(paths)), "/", len(paths))
    return np.concatenate(feats, axis=0)

X_train=embed([resolve_crop(r, train_jsonl) for r in train_rows])
y_train=np.array([r["coarse_class"] for r in train_rows])
X_val=embed([resolve_crop(r, val_jsonl) for r in val_rows])
y_val=np.array([r["coarse_class"] for r in val_rows])

skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_proba=[]; oof_idx=[]; class_order=None
for fold,(tr,va) in enumerate(skf.split(X_train, y_train), start=1):
    clf_fold=LogisticRegression(C=FEATURE_LOGREG_C, class_weight="balanced", max_iter=5000, random_state=42)
    clf_fold.fit(X_train[tr], y_train[tr])
    if class_order is None: class_order=list(clf_fold.classes_)
    assert list(clf_fold.classes_) == class_order
    oof_proba.append(clf_fold.predict_proba(X_train[va])); oof_idx.append(va)
    print("fold", fold, "done")
oof_proba=np.vstack(oof_proba); oof_idx=np.concatenate(oof_idx)
tmp=np.zeros_like(oof_proba); tmp[oof_idx]=oof_proba; oof_proba=tmp
classes=class_order

def hard_preds(probs): return np.array([classes[i] for i in np.argmax(probs, axis=1)])
def lowconf_flash_second_best(probs, threshold):
    preds=[]
    for prob in probs:
        order=np.argsort(prob)[::-1]
        top=classes[order[0]]; conf=float(prob[order[0]])
        if top == "defect_flashover" and conf < threshold:
            top=classes[order[1]]
        preds.append(top)
    return np.array(preds)

threshold_grid=[0.30,0.32,0.33,0.34,0.35,0.36,0.38,0.40,0.45,0.50]
cal=[]
base=hard_preds(oof_proba)
cal.append({"policy":"hard_train_oof","threshold":"","accuracy":accuracy_score(y_train,base),"macro_f1":f1_score(y_train,base,labels=LABELS,average="macro",zero_division=0)})
for t in threshold_grid:
    p=lowconf_flash_second_best(oof_proba,t)
    cal.append({"policy":"flash_lowconf_second_best","threshold":t,"accuracy":accuracy_score(y_train,p),"macro_f1":f1_score(y_train,p,labels=LABELS,average="macro",zero_division=0)})
cal_df=pd.DataFrame(cal)
selected=cal_df[cal_df.policy=="flash_lowconf_second_best"].sort_values(["macro_f1","accuracy"], ascending=False).iloc[0]
selected_t=float(selected.threshold)
print("selected threshold:", selected_t)
display(cal_df.sort_values(["macro_f1","accuracy"], ascending=False))

clf=LogisticRegression(C=FEATURE_LOGREG_C, class_weight="balanced", max_iter=5000, random_state=42)
clf.fit(X_train, y_train)
val_proba=clf.predict_proba(X_val); classes=list(clf.classes_)
policies={"hard_dinov2": hard_preds(val_proba), f"flash_lowconf_second_best_cv{selected_t:.2f}".replace('.','p'): lowconf_flash_second_best(val_proba, selected_t)}
run_dir=REPO_DIR/"outputs"/RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
cal_df.to_csv(run_dir/"train_oof_threshold_selection.csv", index=False)
summary=[]
for name,pred in policies.items():
    out_dir=run_dir/name; out_dir.mkdir(parents=True, exist_ok=True)
    rows=[]
    for row, pr, prob in zip(val_rows, pred, val_proba):
        rec={"record_id":row["record_id"],"gt":row["coarse_class"],"pred_coarse_class":str(pr),"correct":row["coarse_class"]==str(pr),"confidence":float(np.max(prob))}
        for cls,val in zip(classes, prob): rec[f"prob_{cls}"]=float(val)
        rows.append(rec)
    pd.DataFrame(rows).to_csv(out_dir/"val_predictions.csv", index=False)
    conf=pd.DataFrame(confusion_matrix(y_val,pred,labels=LABELS), index=LABELS, columns=LABELS)
    conf.to_csv(out_dir/"confusion_matrix.csv")
    metrics={
        "policy":name,
        "selected_threshold":selected_t,
        "accuracy":float(accuracy_score(y_val,pred)),
        "correct":int((y_val==pred).sum()),
        "total":int(len(y_val)),
        "macro_f1":float(f1_score(y_val,pred,labels=LABELS,average="macro",zero_division=0)),
    }
    for lab in LABELS:
        mask=y_val==lab
        metrics[f"recall_{lab}"]=float((pred[mask]==lab).mean())
    (out_dir/"metrics.json").write_text(json.dumps(metrics,indent=2), encoding="utf-8")
    summary.append(metrics)
summary_df=pd.DataFrame(summary).sort_values(["correct","macro_f1"], ascending=False)
summary_df.to_csv(run_dir/"stage3_dinov2_traincv_policy_summary.csv", index=False)
report="# Stage 3 DINOv2 Train-CV Policy Clean\n\n"+summary_df.to_markdown(index=False)+"\n\n## Train OOF Selection\n\n"+cal_df.sort_values(["macro_f1","accuracy"], ascending=False).to_markdown(index=False)+"\n"
(run_dir/"report.md").write_text(report, encoding="utf-8")
print(report)

package_root=REPO_DIR/"outputs/stage3_dinov2_traincv_policy_package"
if package_root.exists(): shutil.rmtree(package_root)
shutil.copytree(run_dir, package_root)
if ARCHIVE_PATH.exists(): ARCHIVE_PATH.unlink()
sh(["tar","-czf",str(ARCHIVE_PATH),"-C",str(package_root.parent),package_root.name], check=True)
print("Archive:", ARCHIVE_PATH)
shutil.rmtree(REPO_DIR, ignore_errors=True)
